In [1]:
import multiprocessing as mp
import re
import time
from urllib.request import urljoin, urlopen

from bs4 import BeautifulSoup

base_url = "https://morvanzhou.github.io"

# if base_url != "https://morvanzhou.github.io":
#     restricted_crawl = True
# else:
#     restricted_crawl = False

In [2]:
def crawl(url):
    response = urlopen(url)
    time.sleep(0.1)
    return response.read().decode("utf-8")

In [3]:
def parse(html):
    soup = BeautifulSoup(html, "lxml")
    urls = soup.find_all("a", {"href": re.compile(r"^/.+?/$")})
    title = soup.find("h1").get_text().strip()
    page_urls = set([urljoin(base_url, url["href"]) for url in urls])
    url = soup.find("meta", {"property": "og:url"})["content"]
    return title, page_urls, url


In [4]:
unseen = set([
    base_url,
])
seen = set()

count, t1 = 1, time.time()

while len(unseen) != 0:
    # if restricted_crawl and len(seen) > 20:
    #     break
    if len(seen) > 20:
        break

    print("distributed crawling...")
    htmls = [crawl(url) for url in unseen]

    print("distributed parsing...")
    results = [parse(html) for html in htmls]

    print("analyzing...")
    seen.update(unseen)
    unseen.clear()

    for title, page_urls, url in results:
        print(count, title, url)
        count += 1
        unseen.update(page_urls - seen)
print("total time: %.1f s" % (time.time() - t1,))

distributed crawling...
distributed parsing...
analyzing...
1 教程 https://morvanzhou.github.io/
distributed crawling...
distributed parsing...
analyzing...
2 Tensorflow 教程系列 https://morvanzhou.github.io/tutorials/machine-learning/tensorflow/
3 进化算法 Evolutionary Strategies 教程系列 https://morvanzhou.github.io/tutorials/machine-learning/evolutionary-algorithm/
4 Pytorch 教程系列 https://morvanzhou.github.io/tutorials/machine-learning/torch/
5 Threading 多线程教程系列 https://morvanzhou.github.io/tutorials/python-basic/threading/
6 网页爬虫教程系列 https://morvanzhou.github.io/tutorials/data-manipulation/scraping/
7 机器学习实践 https://morvanzhou.github.io/tutorials/machine-learning/ML-practice/
8 Matplotlib 画图教程系列 https://morvanzhou.github.io/tutorials/data-manipulation/plt/
9 关于莫烦 https://morvanzhou.github.io/about/
10 有趣的机器学习系列 https://morvanzhou.github.io/tutorials/machine-learning/ML-intro/
11 Numpy & Pandas 教程系列 https://morvanzhou.github.io/tutorials/data-manipulation/np-pd/
12 数据处理教程系列 https://morvanzhou.gith

In [ ]:
# multiprocessing
unseen = set([
    base_url,
])
seen = set()

pool = mp.Pool(3)
count, t1 = 1, time.time()

while len(unseen) != 0:
    # if restricted_crawl and len(seen) > 20:
    #     break
    if len(seen) > 20:
        break

    print("distributed crawling...")
    crawl_jobs = [pool.apply_async(crawl, args=(url,)) for url in unseen]
    htmls = [j.get() for j in crawl_jobs]

    print("distributed parsing...")
    parse_jobs = [pool.apply_async(parse, args=(html,)) for html in htmls]
    results = [j.get() for j in parse_jobs]

    print("analyzing...")
    seen.update(unseen)
    unseen.clear()

    for title, page_urls, url in results:
        print(count, title, url)
        count += 1
        unseen.update(page_urls - seen)

print("total time: %.1f s" % (time.time() - t1,))

distributed crawling...
